<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Un Experimento A/B en una Plataforma de Aprendizaje Ficticia 🧪
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 03
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/03_Inferencia_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 03: Inferencia Estadística** ([README del módulo](../03%20-%20Inferencia%20Estadistica/README.md)) con un problema **nuevo**: un **experimento A/B** en una plataforma de aprendizaje ficticia que prueba cuatro diseños de la página de inicio (A es el control) y mide la **conversión** (inscripción), el **tiempo en la tarea**, el **dispositivo** usado y la finalización de un piloto.

Al terminar serás capaz de:

1. **Estimar con incertidumbre**: intervalos para una proporción (Wald, Wilson, Clopper-Pearson) con su **cobertura exacta**, el intervalo $t$ de una media y un **bootstrap** percentil para la mediana.
2. **Decidir con una prueba**: la prueba $z$ de dos proporciones, la $t$ de **Welch**, el tamaño del efecto ($d$ de Cohen, $g$ de Hedges), la **potencia** con la $t$ no central, el tamaño de muestra y una **prueba de permutación**.
3. **Comparar frecuencias**: bondad de ajuste, independencia en una tabla de contingencia con **residuos ajustados** y V de Cramér, y qué hacer con **muestras pequeñas** (Yates, Fisher).
4. **Comparar varios grupos**: ANOVA de un factor desde cero con $\eta^2$ y $\omega^2$, la prueba de Levene, **Tukey HSD** y el control de las **comparaciones múltiples** con Holm.

> 📚 **Repasa si lo necesitas:** [Intervalos de confianza](../03%20-%20Inferencia%20Estadistica/00_Intervalos_de_Confianza.ipynb) · [Pruebas de hipótesis, valor p, Z y t](../03%20-%20Inferencia%20Estadistica/01_Pruebas_de_Hipotesis_Valor_P_Z_y_T.ipynb) · [Chi-cuadrado y ANOVA](../03%20-%20Inferencia%20Estadistica/02_Chi_Cuadrado_y_ANOVA.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; la plataforma y sus cifras son **inventadas**). Bibliotecas: `numpy`, `pandas`, `scipy` y `statsmodels` (solo para **contrastar** tus implementaciones). Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

SEMILLA = 2025


def generar_usuarios(semilla=SEMILLA, n_por_variante=500):
    """Experimento ficticio: 500 usuarios por variante de la página (A = control)."""
    rng = np.random.default_rng(semilla)
    conv = {"A": 0.10, "B": 0.12, "C": 0.10, "D": 0.14}                                # probabilidad de inscribirse
    disp = {"A": [0.55, 0.35, 0.10], "B": [0.55, 0.35, 0.10], "C": [0.50, 0.38, 0.12], "D": [0.66, 0.27, 0.07]}
    filas = []
    for v in "ABCD":
        d = rng.choice(["movil", "escritorio", "tableta"], n_por_variante, p=disp[v])
        c = rng.random(n_por_variante) < conv[v]
        filas.append(pd.DataFrame({"variante": v, "dispositivo": d, "convirtio": c.astype(int)}))
    return pd.concat(filas, ignore_index=True)


def generar_panel(semilla=SEMILLA, n_por_variante=40):
    """Subestudio ficticio: tiempo (min) que 40 usuarios por variante tardan en una tarea. Asimétrico a la derecha."""
    rng = np.random.default_rng(semilla + 7)
    medias = {"A": 14.0, "B": 14.5, "C": 14.0, "D": 20.0}
    filas = [pd.DataFrame({"variante": v, "tiempo_min": rng.gamma(5.0, medias[v] / 5.0, n_por_variante).round(2)}) for v in "ABCD"]
    return pd.concat(filas, ignore_index=True)


usuarios = generar_usuarios()
panel = generar_panel()
print(f"Experimento ficticio: {len(usuarios)} usuarios (500 por variante) y un panel de {len(panel)} tiempos en tarea")
print(usuarios.groupby("variante")["convirtio"].agg(["sum", "count"]).T.to_string())

---
## Reto 1: Estimar con intervalos: de la proporción a la mediana 🎯

**Contexto:** quieres intervalos del 95 % para la **conversión** de las variantes A y D, saber qué tan bien cumple cada método lo que promete (**cobertura**), y estimar el tiempo en la tarea de la variante A con un intervalo $t$ y la **mediana** de la D con un *bootstrap*.

**Enunciado**
1. `x_A, n_A, x_D, n_D`: conversiones y usuarios de cada variante en `usuarios` (`groupby`).
2. Tres intervalos **desde cero** para $x$ éxitos de $n$ (con `conf=0.95`, $z=z_{1-\alpha/2}$, $\hat p=x/n$): `ic_wald` $=\hat p\pm z\sqrt{\hat p(1-\hat p)/n}$; `ic_wilson` $=\dfrac{\hat p+\frac{z^2}{2n}\pm z\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}}{1+\frac{z^2}{n}}$; `ic_clopper` con cuantiles beta, $\big(B^{-1}(\tfrac\alpha2;x,n-x+1),\,B^{-1}(1-\tfrac\alpha2;x+1,n-x)\big)$. Wald y Wilson deben aceptar `x` como **arreglo** de NumPy. Guarda `ic_A` (Wilson) e `ic_D` (Wilson) de las variantes.
3. `cobertura_exacta(ic, n, p)`: probabilidad (exacta, sumando la binomial) de que el intervalo `ic` calculado sobre $X\sim\text{Bin}(n,p)$ **contenga** a $p$. Calcula `cob_wald` y `cob_wilson` para `n = 40`, `p = 0.03`.
4. `ic_t(x, conf=0.95)` para la media; `ic_t_A`: intervalo para el tiempo medio de la variante A en `panel`.
5. `ic_boot(x, estadistico, B, rng)`: *bootstrap* percentil (matriz de `B` remuestras con reemplazo, vectorizada); `ic_med_D`: intervalo para la **mediana** del tiempo de la variante D con `B = 5000`.

**Criterios de aceptación:** coinciden con `proportion_confint` y `stats.t.interval`; Wald cubre muy por debajo del 95 % con `p = 0.03`; el bootstrap contiene a la mediana y su ancho es parecido al de `stats.bootstrap`.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: completa
x_A = n_A = x_D = n_D = None


def ic_wald(x, n, conf=0.95):
    # TODO: devuelve (inferior, superior)
    return None, None


def ic_wilson(x, n, conf=0.95):
    # TODO
    return None, None


def ic_clopper(x, n, conf=0.95):
    # TODO: con stats.beta.ppf (supón 0 < x < n)
    return None, None


ic_A = ic_D = None                 # intervalos de Wilson de las conversiones A y D


def cobertura_exacta(ic, n, p, conf=0.95):
    # TODO: suma P(X = k) de los k = 0..n cuyo intervalo ic(k, n, conf) contiene a p
    return None


cob_wald = cob_wilson = None       # n = 40, p = 0.03


def ic_t(x, conf=0.95):
    # TODO: intervalo t para la media
    return None, None


tiempo_A = panel.loc[panel["variante"] == "A", "tiempo_min"].to_numpy()
tiempo_D = panel.loc[panel["variante"] == "D", "tiempo_min"].to_numpy()
ic_t_A = None


def ic_boot(x, estadistico, B, rng, conf=0.95):
    # TODO: bootstrap percentil; `estadistico` se aplica a cada fila (axis=1) de la matriz de remuestras
    return None, None


rng = np.random.default_rng(SEMILLA + 1)
ic_med_D = None                    # intervalo bootstrap de la mediana del tiempo en D

In [ ]:
# Autoverificación del Reto 1 (no modificar)
_g = usuarios.groupby("variante")["convirtio"].agg(["sum", "count"])
assert (x_A, n_A, x_D, n_D) == (_g.loc["A", "sum"], _g.loc["A", "count"], _g.loc["D", "sum"], _g.loc["D", "count"]), "x_A, n_A, x_D, n_D: suma y conteo de `convirtio` por variante (groupby)."
_xs = np.array([17, 48, 67, 250])
for _nom, _f, _m in (("ic_wald", ic_wald, "normal"), ("ic_wilson", ic_wilson, "wilson")):
    _lo, _hi = _f(_xs, 500)
    _el, _eh = proportion_confint(_xs, 500, method=_m)
    assert _lo is not None and np.allclose(_lo, _el, atol=1e-9) and np.allclose(_hi, _eh, atol=1e-9), f"{_nom} debe coincidir con proportion_confint(method='{_m}') y aceptar arreglos (usa np.asarray(x) / n)."
_lo, _hi = ic_clopper(48, 500); _el, _eh = proportion_confint(48, 500, method="beta")
assert _lo is not None and abs(_lo - _el) < 1e-9 and abs(_hi - _eh) < 1e-9, "ic_clopper: stats.beta.ppf(alfa/2, x, n - x + 1) y stats.beta.ppf(1 - alfa/2, x + 1, n - x)."
assert np.allclose(ic_A, proportion_confint(x_A, n_A, method="wilson")) and np.allclose(ic_D, proportion_confint(x_D, n_D, method="wilson")), "ic_A e ic_D son los intervalos de Wilson de cada variante."
def _cob(m, n, p):
    k = np.arange(n + 1); lo, hi = proportion_confint(k, n, method=m)
    return stats.binom.pmf(k, n, p)[(lo <= p) & (p <= hi)].sum()
assert cob_wald is not None and abs(cob_wald - _cob("normal", 40, 0.03)) < 1e-9 and abs(cob_wilson - _cob("wilson", 40, 0.03)) < 1e-9, "cobertura_exacta: recorre k = 0..n, calcula el intervalo de cada k, y suma binom.pmf(k, n, p) de los que contienen a p."
assert cob_wald < 0.80 and cob_wilson > cob_wald + 0.15, "Con p = 0.03 y n = 40, Wald cubre muy por debajo del 95 % y Wilson mucho más."
_t = stats.t.interval(0.95, len(tiempo_A) - 1, loc=tiempo_A.mean(), scale=stats.sem(tiempo_A))
assert ic_t_A is not None and np.allclose(ic_t_A, _t, atol=1e-9), "ic_t: media ± t(1 - alfa/2, n-1) · s / sqrt(n), con s de ddof=1."
_b = stats.bootstrap((tiempo_D,), np.median, confidence_level=0.95, n_resamples=5000, method="percentile", random_state=1).confidence_interval
assert ic_med_D is not None and ic_med_D[0] < np.median(tiempo_D) < ic_med_D[1], "El intervalo bootstrap debe contener a la mediana observada."
assert 0.75 < (ic_med_D[1] - ic_med_D[0]) / (_b.high - _b.low) < 1.33, "El ancho de tu intervalo bootstrap debería ser parecido al de stats.bootstrap: remuestrea con reemplazo filas de tamaño n y calcula la mediana de cada una (axis=1)."
_r = ic_boot(tiempo_A, np.mean, 2000, np.random.default_rng(3))
assert _r[0] < tiempo_A.mean() < _r[1] and abs((_r[1] - _r[0]) / (ic_t_A[1] - ic_t_A[0]) - 1) < 0.15, "ic_boot debe servir con cualquier estadístico: para la media su ancho debe parecerse al del intervalo t."
print(f"✅ Reto 1 superado: Wilson A {np.round(ic_A, 3)}, D {np.round(ic_D, 3)} | cobertura (n=40, p=0.03): Wald {cob_wald:.3f}, Wilson {cob_wilson:.3f} | IC t del tiempo en A {np.round(ic_t_A, 2)} | IC bootstrap de la mediana en D {np.round(ic_med_D, 2)}")

---
## Reto 2: ¿Es mejor la variante D que el control? 🧪

**Contexto:** compara D con A en dos métricas: la **conversión** (proporciones) y el **tiempo en la tarea** (medias, datos asimétricos). Además de los valores $p$ debes cuantificar el **tamaño del efecto**, la **potencia** del estudio y cuántos usuarios haría falta tener.

**Enunciado** (usa `x_A, n_A, x_D, n_D`, `tiempo_A` y `tiempo_D` del reto anterior)
1. `z_dos_prop(x1, n1, x2, n2)` → `(z, p)`: prueba $z$ de dos proporciones con varianza **combinada** $\hat p=(x_1+x_2)/(n_1+n_2)$, $z=\dfrac{\hat p_2-\hat p_1}{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}$, valor $p$ bilateral. Guarda `z_conv, p_conv` para D frente a A (A es el grupo 1).
2. `welch(x, y)` → `(t, gl, p)`: prueba $t$ de Welch con $t=\dfrac{\bar x-\bar y}{\sqrt{s_x^2/n_x+s_y^2/n_y}}$, los grados de libertad de Welch-Satterthwaite y valor $p$ bilateral. Guarda `t_w, gl_w, p_w` aplicándola como `welch(tiempo_A, tiempo_D)`.
3. `cohen_d(x, y)` (con la desviación combinada) y `hedges_g(x, y)` $=d\left(1-\dfrac{3}{4(n_x+n_y)-9}\right)$; guarda `d_obs` y `g_obs` para `(tiempo_D, tiempo_A)`.
4. `potencia_t(d, n, alfa=0.05)`: potencia bilateral de la prueba $t$ de dos muestras independientes con `n` por grupo, usando la $t$ **no central** ($\delta=d\sqrt{n/2}$, gl $=2n-2$). `pot_obs` $=$ `potencia_t(abs(d_obs), 40)`. `n_80`: menor `n` por grupo con potencia $\ge0.80$ para $d=0.5$.
5. `p_perm`: valor $p$ de una **prueba de permutación** para la diferencia de medias `tiempo_D - tiempo_A` (`B = 9999` permutaciones; $p=(1+\#\{|d^\ast|\ge|d_{obs}|\})/(B+1)$).

**Criterios de aceptación:** `z`, `t` y sus valores $p$ coinciden con statsmodels/SciPy; la potencia coincide con `TTestIndPower`; `n_80` es el entero mínimo; y el valor $p$ de permutación se parece al de Welch.

**Puntaje sugerido:** 25 puntos.

In [ ]:
def z_dos_prop(x1, n1, x2, n2):
    # TODO: devuelve (z, p bilateral) con la varianza combinada
    return None, None


z_conv = p_conv = None


def welch(x, y):
    # TODO: devuelve (t, gl, p bilateral)
    return None, None, None


t_w = gl_w = p_w = None


def cohen_d(x, y):
    # TODO: (media de x - media de y) / desviación combinada
    return None


def hedges_g(x, y):
    # TODO: corrección de d por muestras pequeñas
    return None


d_obs = g_obs = None


def potencia_t(d, n, alfa=0.05):
    # TODO: potencia bilateral con la t no central
    return None


pot_obs = None
n_80 = None             # menor n por grupo con potencia >= 0.80 si d = 0.5

rng = np.random.default_rng(SEMILLA + 2)
p_perm = None

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert x_A is not None and n_D is not None, "Primero resuelve el Reto 1: aquí necesitas x_A, n_A, x_D y n_D."
_z, _p = proportions_ztest([x_D, x_A], [n_D, n_A])
assert z_conv is not None and abs(z_conv - _z) < 1e-9 and abs(p_conv - _p) < 1e-9, "z_dos_prop: varianza combinada p = (x1+x2)/(n1+n2); z = (p2 - p1)/sqrt(p(1-p)(1/n1 + 1/n2)); p = 2·norm.sf(|z|)."
_w = stats.ttest_ind(tiempo_A, tiempo_D, equal_var=False)
assert t_w is not None and abs(t_w - _w.statistic) < 1e-9 and abs(p_w - _w.pvalue) < 1e-9 and abs(gl_w - _w.df) < 1e-6, "welch: t = (media x - media y)/sqrt(sx²/nx + sy²/ny); gl = (vx + vy)² / (vx²/(nx-1) + vy²/(ny-1)) con vx = sx²/nx; p = 2·t.sf(|t|, gl)."
_sp = math.sqrt((39 * tiempo_A.var(ddof=1) + 39 * tiempo_D.var(ddof=1)) / 78); _d = (tiempo_D.mean() - tiempo_A.mean()) / _sp
assert d_obs is not None and abs(d_obs - _d) < 1e-9 and abs(g_obs - _d * (1 - 3 / (4 * 80 - 9))) < 1e-9 and 0 < g_obs < d_obs, "cohen_d = diferencia de medias / desviación combinada; hedges_g = d·(1 - 3/(4(nx+ny) - 9)) y debe ser un poco menor que d."
_pw = TTestIndPower()
for _dd, _nn in ((0.2, 30), (0.5, 40), (0.8, 25)):
    assert abs(potencia_t(_dd, _nn) - _pw.power(_dd, nobs1=_nn, alpha=0.05)) < 1e-6, f"potencia_t({_dd}, {_nn}) debe coincidir con TTestIndPower().power: nct.sf(tcrit, gl, nc) + nct.cdf(-tcrit, gl, nc), con nc = d·sqrt(n/2)."
assert pot_obs is not None and abs(pot_obs - _pw.power(abs(_d), nobs1=40, alpha=0.05)) < 1e-6, "pot_obs = potencia_t(abs(d_obs), 40)."
assert n_80 == math.ceil(_pw.solve_power(0.5, power=0.8, alpha=0.05)) == 64, "n_80: el menor n entero con potencia >= 0.80 para d = 0.5 (debe salir 64 por grupo)."
assert p_perm is not None and 0 < p_perm <= 1 and abs(p_perm - _w.pvalue) < 0.04, "p_perm: mezcla los datos (rng.permutation), recalcula la diferencia de medias 9999 veces y usa (1 + #{|d*| >= |d_obs|}) / (B + 1)."
print(f"✅ Reto 2 superado: conversión z = {z_conv:.2f} (p = {p_conv:.3f}) | tiempo: t = {t_w:.2f}, gl = {gl_w:.1f}, p = {p_w:.4f}, permutación p = {p_perm:.4f} | d = {d_obs:.2f}, g = {g_obs:.2f} | potencia observada {pot_obs:.2f}; para d = 0.5 hacen falta {n_80} por grupo")

---
## Reto 3: ¿Cambia el dispositivo según la variante? 📊

**Contexto:** la variante D parece atraer más usuarios móviles. Contrasta esa impresión con una prueba de **independencia** entre `variante` y `dispositivo`, verifica si el mix de dispositivos del control coincide con el esperado, y decide qué prueba usar en un **piloto pequeño** (14 usuarios).

**Enunciado**
1. `tabla`: tabla de contingencia (`pd.crosstab`) de `variante` por `dispositivo`.
2. `chi2_indep(tabla)` → diccionario con `chi2`, `gl`, `p`, `esperados` (arreglo) y `v` (V de Cramér), sin corrección de continuidad. Guarda `res = chi2_indep(tabla)` y los **residuos estandarizados ajustados** `res_aj` $=\dfrac{O-E}{\sqrt{E\,(1-r_i/n)(1-c_j/n)}}$ (arreglo del tamaño de la tabla), con $r_i$ y $c_j$ los totales de fila y columna.
3. `chi2_gof(obs, p_esp)` → `(chi2, gl, p)`. Aplícalo a la fila de la variante A con `P_ESPERADO` $=(0.35, 0.55, 0.10)$ (35 % escritorio, 55 % móvil, 10 % tableta, en el orden de las columnas de `tabla`: `escritorio`, `movil`, `tableta`): guarda `gof_A`.
4. Piloto `tabla_pil = [[7, 1], [2, 4]]` (filas: diseño nuevo y viejo; columnas: completó / abandonó): `esp_min` (menor esperado), `p_sin` (chi-cuadrado **sin** corrección), `p_yates` (con corrección) y `p_fisher` (prueba exacta de Fisher), y `conclusion`: `'rechazar'` si `p_fisher < 0.05`, o `'no rechazar'`.

**Criterios de aceptación:** coinciden con `stats.chi2_contingency` y `stats.chisquare`; el piloto tiene un esperado menor que 5 y las tres pruebas no concuerdan en la decisión; y la `conclusion` sigue a Fisher.

**Puntaje sugerido:** 20 puntos.

In [ ]:
P_ESPERADO = np.array([0.35, 0.55, 0.10])        # escritorio, movil, tableta (orden alfabético de las columnas)
tabla_pil = np.array([[7, 1], [2, 4]])

# TODO: completa
tabla = None


def chi2_indep(tabla):
    # TODO: {'chi2': ..., 'gl': ..., 'p': ..., 'esperados': arreglo, 'v': V de Cramér}
    return None


res = None
res_aj = None            # residuos estandarizados ajustados (arreglo del tamaño de la tabla)


def chi2_gof(obs, p_esp):
    # TODO: (chi2, gl, p) de la bondad de ajuste
    return None, None, None


gof_A = None

esp_min = p_sin = p_yates = p_fisher = None
conclusion = None        # 'rechazar' o 'no rechazar' según Fisher al 5 %

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_t = pd.crosstab(usuarios["variante"], usuarios["dispositivo"])
assert tabla is not None and tabla.shape == (4, 3) and np.array_equal(tabla.to_numpy(), _t.to_numpy()), "tabla = pd.crosstab(usuarios['variante'], usuarios['dispositivo']) (4 variantes × 3 dispositivos)."
_c = stats.chi2_contingency(_t, correction=False)
assert res is not None and abs(res["chi2"] - _c[0]) < 1e-9 and res["gl"] == _c[2] == 6 and abs(res["p"] - _c[1]) < 1e-12 and np.allclose(res["esperados"], _c[3]), "chi2_indep: esperados = total fila × total columna / n; chi2 = Σ (O - E)²/E; gl = (filas-1)(columnas-1); p = chi2.sf(chi2, gl)."
assert abs(res["v"] - stats.contingency.association(_t, method="cramer", correction=False)) < 1e-9, "v = sqrt(chi2 / (n · min(filas-1, columnas-1)))."
_tt = _t.to_numpy().astype(float); _nn = _tt.sum(); _E = _c[3]
_aj = (_tt - _E) / np.sqrt(_E * (1 - _tt.sum(1, keepdims=True) / _nn) * (1 - _tt.sum(0, keepdims=True) / _nn))
assert res_aj is not None and np.allclose(res_aj, _aj) and np.allclose((res_aj ** 2).sum() > 0, True), "res_aj = (O - E) / sqrt(E · (1 - total fila/n) · (1 - total columna/n)) con arreglos que se transmiten (keepdims=True)."
_g = stats.chisquare(_t.loc["A"].to_numpy(), 500 * P_ESPERADO)
assert gof_A is not None and abs(gof_A[0] - _g.statistic) < 1e-9 and gof_A[1] == 2 and abs(gof_A[2] - _g.pvalue) < 1e-12, "chi2_gof: esperados = n · p_esp; chi2 = Σ (O - E)²/E; gl = categorías - 1; p = chi2.sf."
_e = tabla_pil.sum(1, keepdims=True) * tabla_pil.sum(0, keepdims=True) / tabla_pil.sum()
assert esp_min is not None and abs(esp_min - _e.min()) < 1e-12 and esp_min < 5, "esp_min es el menor de los esperados (total fila × total columna / n): aquí es menor que 5, así que chi-cuadrado no es confiable."
assert abs(p_sin - stats.chi2_contingency(tabla_pil, correction=False)[1]) < 1e-12 and abs(p_yates - stats.chi2_contingency(tabla_pil, correction=True)[1]) < 1e-12 and abs(p_fisher - stats.fisher_exact(tabla_pil)[1]) < 1e-12, "p_sin y p_yates con chi2_contingency(correction=False/True); p_fisher con stats.fisher_exact (toma el segundo valor)."
assert p_sin < 0.05 < p_yates and p_fisher > 0.05, "En este piloto la chi-cuadrado sin corrección rechaza, pero Yates y Fisher no: la decisión cambia."
assert conclusion == "no rechazar", "conclusion sigue a Fisher al 5 %: con p_fisher > 0.05 es 'no rechazar'."
print(f"✅ Reto 3 superado: chi2 = {res['chi2']:.2f} (gl 6, p = {res['p']:.4f}), V = {res['v']:.3f}, mayor |residuo ajustado| = {np.abs(res_aj).max():.2f} | mix de A: p = {gof_A[2]:.3f} | piloto: p sin {p_sin:.3f}, Yates {p_yates:.3f}, Fisher {p_fisher:.3f} → {conclusion}")

---
## Reto 4: Cuatro variantes, un ANOVA y muchas comparaciones 🧮

**Contexto:** el `panel` tiene el tiempo en la tarea de 40 usuarios por variante. Antes de comparar de dos en dos debes saber si **alguna** variante difiere (ANOVA), si el supuesto de varianzas iguales es razonable (Levene), **cuáles** difieren controlando el error global (Tukey) y qué pasa si se hacen 6 pruebas $t$ separadas sin corregir (o con Holm).

**Enunciado**
1. `grupos`: lista de arreglos `tiempo_min` de las variantes `A`, `B`, `C`, `D` (en ese orden).
2. `anova_uno(grupos)` → diccionario con `ssb`, `ssw`, `sst`, `f`, `p`, `eta2` $=SSB/SST$ y `omega2` $=\dfrac{SSB-(k-1)\,MSW}{SST+MSW}$. Guarda `anova = anova_uno(grupos)`.
3. `p_levene`: valor $p$ de la prueba de Levene (`stats.levene(..., center='median')`).
4. `pares_tukey`: lista ordenada de pares `(i, j)` con $i<j$ (índices 0 a 3) cuya diferencia es significativa al 5 % según `stats.tukey_hsd(*grupos)`.
5. `p_pares`: arreglo con los 6 valores $p$ de Welch (`stats.ttest_ind(..., equal_var=False)`) de todos los pares `itertools.combinations(range(4), 2)`. `holm(p)`: valores $p$ **ajustados** por Holm (desde cero): ordena de menor a mayor, $\tilde p_{(i)}=\max_{j\le i}\{(m-j+1)\,p_{(j)}\}$ acotado en 1, y devuélvelos en el orden original. `rech_sin` (cuántos pares tienen $p<0.05$ sin corregir) y `rech_holm` (cuántos con Holm).

**Criterios de aceptación:** el ANOVA coincide con `stats.f_oneway`; $\eta^2>\omega^2$; Holm coincide con `multipletests` y nunca rechaza más que sin corregir; y `pares_tukey` coincide con un cálculo independiente.

**Puntaje sugerido:** 25 puntos.

In [ ]:
import itertools

# TODO: completa
grupos = None                         # [arreglo de A, de B, de C, de D]


def anova_uno(grupos):
    # TODO: {'ssb', 'ssw', 'sst', 'f', 'p', 'eta2', 'omega2'}
    return None


anova = None
p_levene = None
pares_tukey = None                    # lista de pares (i, j), i < j, con p < 0.05


def holm(p):
    # TODO: valores p ajustados por Holm, en el orden original
    return None


p_pares = None                        # 6 valores p de Welch, en el orden de itertools.combinations(range(4), 2)
p_holm = None
rech_sin = rech_holm = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_gr = [panel.loc[panel["variante"] == v, "tiempo_min"].to_numpy() for v in "ABCD"]
assert grupos is not None and len(grupos) == 4 and all(np.array_equal(a, b) for a, b in zip(grupos, _gr)), "grupos: lista con el arreglo de tiempo_min de A, B, C y D en ese orden."
_f = stats.f_oneway(*_gr)
assert anova is not None and abs(anova["f"] - _f.statistic) < 1e-9 and abs(anova["p"] - _f.pvalue) < 1e-12, "F = (SSB/(k-1)) / (SSW/(N-k)); SSB = Σ n_i (media_i - media global)²; SSW = Σ Σ (x - media_i)²; p = f.sf(F, k-1, N-k)."
assert abs(anova["ssb"] + anova["ssw"] - anova["sst"]) < 1e-8 and abs(anova["sst"] - ((np.concatenate(_gr) - np.concatenate(_gr).mean()) ** 2).sum()) < 1e-6, "Debe cumplirse SST = SSB + SSW, y SST = Σ (x - media global)²."
_msw = anova["ssw"] / (160 - 4)
assert abs(anova["eta2"] - anova["ssb"] / anova["sst"]) < 1e-12 and abs(anova["omega2"] - (anova["ssb"] - 3 * _msw) / (anova["sst"] + _msw)) < 1e-12 and anova["eta2"] > anova["omega2"] > 0, "eta2 = SSB/SST; omega2 = (SSB - (k-1)·MSW)/(SST + MSW), y debe ser menor que eta2."
assert p_levene is not None and abs(p_levene - stats.levene(*_gr, center="median").pvalue) < 1e-12, "p_levene = stats.levene(*grupos, center='median').pvalue."
_th = stats.tukey_hsd(*_gr)
assert pares_tukey == [(i, j) for i, j in itertools.combinations(range(4), 2) if _th.pvalue[i, j] < 0.05], "pares_tukey: los (i, j) con i < j para los que tukey_hsd(*grupos).pvalue[i, j] < 0.05, en orden."
assert len(pares_tukey) >= 1, "Con este panel al menos una pareja (D frente a otra) debe resultar distinta."
_pp = np.array([stats.ttest_ind(_gr[i], _gr[j], equal_var=False).pvalue for i, j in itertools.combinations(range(4), 2)])
assert p_pares is not None and np.allclose(p_pares, _pp), "p_pares: stats.ttest_ind(grupos[i], grupos[j], equal_var=False).pvalue en el orden de itertools.combinations(range(4), 2)."
_ph = multipletests(_pp, method="holm")[1]
assert np.allclose(holm(_pp), _ph) and np.allclose(holm([0.01, 0.04, 0.03, 0.005]), multipletests([0.01, 0.04, 0.03, 0.005], method="holm")[1]), "holm: ordena, multiplica por (m - posición), toma el máximo acumulado (np.maximum.accumulate), acota en 1 y vuelve al orden original."
assert np.all(holm(_pp) >= _pp - 1e-12) and p_holm is not None and np.allclose(p_holm, _ph), "Los valores ajustados nunca son menores que los originales; guarda p_holm = holm(p_pares)."
assert rech_sin == int((_pp < 0.05).sum()) and rech_holm == int((_ph < 0.05).sum()) and rech_holm <= rech_sin, "rech_sin y rech_holm cuentan los p < 0.05 sin y con la corrección; con Holm nunca hay más rechazos."
print(f"✅ Reto 4 superado: F = {anova['f']:.2f} (p = {anova['p']:.4f}), eta² = {anova['eta2']:.3f}, omega² = {anova['omega2']:.3f} | Levene p = {p_levene:.3f} | Tukey: {pares_tukey} | rechazos sin corregir {rech_sin}, con Holm {rech_holm}")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué intervalo usarías para la conversión de A y por qué? Con `cob_wald` y `cob_wilson`, ¿qué promete y qué cumple cada método cuando $p$ es pequeño? ¿Qué dice (y qué **no** dice) el «95 %»?
- Para D frente a A, compara `p_conv` y `p_w`: ¿qué concluyes en cada métrica? ¿Qué te dicen `d_obs`, `pot_obs` y `n_80` sobre si el estudio de conversión tenía potencia suficiente? ¿Es lo mismo «no significativo» que «sin efecto»?
- ¿Por qué `p_perm` se parece a `p_w` y cuándo preferirías la permutación? ¿Qué supuesto evita?
- En el piloto, ¿por qué la decisión cambia entre `p_sin`, `p_yates` y `p_fisher`, y cuál te convence? ¿Qué muestran los residuos ajustados de la tabla de dispositivos y qué indica la V de Cramér sobre la **relevancia**?
- ¿Qué aporta Tukey o Holm frente a las seis pruebas $t$ sueltas? Relaciona `rech_sin` y `rech_holm` con el riesgo de falsos descubrimientos y con el tamaño del efecto (`eta2`, `omega2`).

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: intervalos de proporción y su cobertura, intervalo $t$ y bootstrap | 25 |
| Reto 2: prueba $z$, Welch, tamaño del efecto, potencia, tamaño de muestra y permutación | 25 |
| Reto 3: chi-cuadrado de independencia y de ajuste, residuos y muestras pequeñas | 20 |
| Reto 4: ANOVA, $\eta^2$/$\omega^2$, Levene, Tukey y Holm | 15 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Interpreté los valores p y los intervalos sin decir «probabilidad de que la hipótesis sea cierta».
- [ ] Reporté tamaño del efecto y potencia, no solo si «salió significativo».
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>